# Notebook 01b — Confound and Leakage Audit

**Project:** A Multi-Modal Deep Learning Approach for Eggshell Micro-Crack Morphology Analysis and Root-Cause Diagnosis in Poultry Breeder Farms
**Author:** M.P.N. Sharindi (Coventry Index 16110429)
**Stage:** 01b of 12 — runs after notebook 01, before notebook 02

---

### Why this notebook exists

Notebook 01 found that **five of seven capture variables differ significantly between the four
classes after Holm correction**, four of them with effect sizes above the "moderate" threshold:

| Capture variable | epsilon squared | |
|---|---|---|
| focus (Laplacian variance inside egg) | 0.143 | large |
| image size (px) | 0.120 | moderate |
| luminance SD inside egg | 0.089 | moderate |
| red/blue ratio | 0.087 | moderate |
| mean luminance inside egg | 0.070 | moderate |

It also found the likely cause. Eggs were not uploaded in random order: **43.8 % of all Intact eggs
fall in the first quartile of upload order, and not a single Star egg does** (chi-square = 158.7,
p = 1.4e-29, Cramer's V = 0.285). Capture happened in sessions, and the sessions were blocked by
class. Lighting, distance and focus drift between sessions, so class became correlated with
acquisition conditions.

**This is the single most serious threat to the validity of this research, and it must be resolved
before any model is trained.** If it is not, a convolutional network can reach a high score by
reading the capture session rather than the crack, the held-out test score will be inflated, and
the result will not transfer to a new farm. An examiner who spots this unaddressed can discount the
entire evaluation.

### What this notebook decides

| Test | Question | Decision it drives |
|---|---|---|
| A | How well can a model predict the class from capture metadata **alone**, with no pixels? | The size of the shortcut. This is the headline number |
| B | Is the focus difference a capture artefact, or is it the crack itself being high-frequency texture? | Whether `lap_var_egg` is a confound or genuine signal |
| C | Does capture order structure the dataset into sessions? | Whether the split in notebook 02 must be session-aware |
| D | Do letterbox padding and aspect ratio differ by class? | Pre-processing and padding policy |
| E | Which normalisation actually removes the shortcut? | The pre-processing decision in notebook 02 |

### Reading the result of Test A

Chance-level macro F1 for four classes at this imbalance is about 0.21 (always predicting Intact).

| Metadata-only macro F1 | Verdict |
|---|---|
| below 0.30 | negligible; proceed as planned |
| 0.30 – 0.45 | real but manageable; neutralise in pre-processing and report |
| 0.45 – 0.60 | serious; normalisation is mandatory and the limitation is reported prominently |
| above 0.60 | severe; a metadata-matched subset analysis is required as a sensitivity check |

Be prepared for a number in the third band. That is survivable and honest. Hiding it is not.

---
## 0. Configuration and environment

In [ ]:
import os
os.environ["EGG_ARTEFACT_ROOT"] = r"E:\Degree\Final Research\Egg_Crack_Dataset\Eggshell-Diagnostic-Research\Artefact"

N_JOBS = -1
FIG_DPI = 300
POLY_DILATE_FRAC = 0.03   # dilate the crack mask by 3% of image side before excluding it

In [ ]:
import sys, json, math, warnings, platform
from pathlib import Path
from datetime import datetime

_here = Path.cwd()
_src = next((p / "src" for p in [_here, *_here.parents] if (p / "src" / "common").is_dir()), None)
if _src is None:
    raise RuntimeError("Run this from Artefact/notebooks/ or set EGG_ARTEFACT_ROOT.")
sys.path.insert(0, str(_src))

import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from scipy import stats
from joblib import Parallel, delayed
from common import project_config as CFG

warnings.filterwarnings("ignore", category=FutureWarning)
CFG.seed_everything()

STAGE = "01b_confound_audit"
OUT = CFG.results_dir(STAGE)
FIG_DIR, TAB_DIR = OUT / "figures", OUT / "tables"
PROC = CFG.DATA_DIR / "processed"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": FIG_DPI, "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})
_N = {"i": 0}; FIGS, TABS = [], []
def savefig(fig, name, caption=""):
    _N["i"] += 1; f = f"{_N['i']:02d}_{name}.png"
    fig.savefig(FIG_DIR / f, bbox_inches="tight"); FIGS.append({"file": f, "caption": caption})
    print(f"  [figure] {f}")
def savetable(df, name, caption=""):
    p = TAB_DIR / f"{name}.csv"; df.to_csv(p)
    TABS.append({"file": p.name, "caption": caption, "shape": list(df.shape)})
    print(f"  [table ] {p.name}  {df.shape}"); return df

S = {"stage": STAGE, "generated_at": datetime.now().isoformat(timespec="seconds")}

master = pd.read_csv(PROC / "master_table.csv")

def resolve_image_paths(df: pd.DataFrame) -> pd.DataFrame:
    """Make sure a usable absolute image path exists, whatever the table calls it."""
    if "path" in df.columns and df["path"].notna().all() and Path(str(df["path"].iloc[0])).is_file():
        return df
    for cand in ("path", "file_name", "filename", "file"):
        if cand in df.columns:
            p = df[cand].astype(str).map(lambda v: Path(v) if Path(v).is_absolute()
                                         else CFG.IMAGES_DIR / Path(v).name)
            if p.map(lambda q: q.is_file()).all():
                df["path"] = p.astype(str)
                print(f"image paths resolved from column '{cand}'")
                return df
    df["path"] = df["egg_id"].map(lambda e: str(CFG.IMAGES_DIR / f"{e}.jpg"))
    missing = int(sum(not Path(q).is_file() for q in df["path"]))
    print(f"image paths rebuilt from egg_id; missing files: {missing}")
    if missing:
        raise FileNotFoundError(f"{missing} images could not be located under {CFG.IMAGES_DIR}")
    return df

master = resolve_image_paths(master)
master["label4"] = pd.Categorical(master["label4"], categories=CFG.LABEL4, ordered=True)
master["age_class"] = pd.Categorical(master["age_class"], categories=CFG.AGE_ORDER, ordered=True)
# capture order is encoded in the egg id, which was assigned by capture order
master["capture_order"] = master["egg_id"].str.extract(r"E(\d{3})").astype(int)
print(f"master table: {master.shape[0]} rows x {master.shape[1]} columns")
print(f"torch available: ", end="")
try:
    import torch; print(torch.__version__, "cuda", torch.cuda.is_available())
except ImportError:
    print("not installed (not needed here)")

---
## Test A — How large is the shortcut?

A gradient-boosted tree is trained on capture metadata only: image geometry, exposure, focus and
colour. It never sees a pixel of the egg and has no access to crack geometry. Whatever accuracy it
reaches is accuracy a convolutional network could also obtain **without looking at the crack**.

Three feature sets are compared, because they map onto three different pre-processing policies:

* **all capture features** — the worst case, what a model sees if nothing is normalised
* **resize-invariant** — drops absolute size and file size, simulating the fact that every image is
  resized to a fixed input resolution, which destroys absolute-resolution information
* **resize + photometric-invariant** — additionally drops absolute brightness and colour, simulating
  per-image intensity standardisation

If performance collapses between the second and third set, per-image normalisation is the fix. If it
does not, the shortcut is structural and needs a stronger response.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier

SIZE_FEATS  = ["min_side", "megapixels", "file_kb", "width", "height"]
PHOTO_FEATS = ["mean_lum", "std_lum", "p01_lum", "p99_lum", "egg_mean_lum", "egg_std_lum",
               "mean_B", "mean_G", "mean_R", "rb_ratio"]
SHAPE_FEATS = ["egg_area_frac", "egg_bbox_fill"]
OTHER_FEATS = ["frac_clipped_hi", "frac_clipped_lo", "egg_frac_clipped_hi",
               "lap_var_global", "lap_var_egg"]

FEATURE_SETS = {
    "all capture features": SIZE_FEATS + PHOTO_FEATS + SHAPE_FEATS + OTHER_FEATS,
    "resize-invariant": PHOTO_FEATS + SHAPE_FEATS + OTHER_FEATS,
    "resize + photometric-invariant": SHAPE_FEATS + ["frac_clipped_hi", "frac_clipped_lo"],
}
FEATURE_SETS = {k: [c for c in v if c in master.columns] for k, v in FEATURE_SETS.items()}

def shortcut_probe(feats, y, label, n_splits=5):
    X = master[feats].to_numpy(dtype=float)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=CFG.RANDOM_SEED)
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08,
                                         random_state=CFG.RANDOM_SEED)
    yp = cross_val_predict(clf, X, y, cv=cv, n_jobs=1)
    dummy = cross_val_predict(DummyClassifier(strategy="most_frequent"), X, y, cv=cv)
    return {
        "feature_set": label, "n_features": len(feats),
        "macro_f1": round(float(f1_score(y, yp, average="macro")), 3),
        "balanced_acc": round(float(balanced_accuracy_score(y, yp)), 3),
        "accuracy": round(float((yp == y).mean()), 3),
        "baseline_macro_f1": round(float(f1_score(y, dummy, average="macro")), 3),
        "_pred": yp,
    }

y4 = master["label4"].astype(str).to_numpy()
y2 = master["is_cracked"].to_numpy()

rows4 = [shortcut_probe(f, y4, k) for k, f in FEATURE_SETS.items()]
rows2 = [shortcut_probe(f, y2, k) for k, f in FEATURE_SETS.items()]

probe4 = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in rows4])
probe2 = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in rows2])
probe4.insert(0, "task", "4-class morphology"); probe2.insert(0, "task", "binary detection")
probe = pd.concat([probe4, probe2], ignore_index=True).set_index(["task", "feature_set"])
savetable(probe, "t01_metadata_shortcut_probe",
          "Classification performance from capture metadata alone, with no image content")
S["shortcut_probe"] = probe.reset_index().to_dict("records")
print(probe.to_string())

headline = rows4[0]["macro_f1"]
band = ("negligible" if headline < 0.30 else "real but manageable" if headline < 0.45
        else "serious" if headline < 0.60 else "severe")
S["shortcut_headline_macro_f1"] = headline
S["shortcut_band"] = band
print(f"\n>>> HEADLINE: macro F1 = {headline} from metadata alone  -->  {band.upper()}")

In [ ]:
# --- which capture variables carry the shortcut? -------------------------
feats = FEATURE_SETS["all capture features"]
X = master[feats].to_numpy(dtype=float)
clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08,
                                     random_state=CFG.RANDOM_SEED).fit(X, y4)
pi = permutation_importance(clf, X, y4, n_repeats=20, random_state=CFG.RANDOM_SEED,
                            scoring="f1_macro", n_jobs=N_JOBS)
imp = (pd.DataFrame({"feature": feats, "importance": pi.importances_mean,
                     "sd": pi.importances_std})
       .sort_values("importance", ascending=False).set_index("feature"))
savetable(imp, "t02_shortcut_feature_importance",
          "Permutation importance of capture variables for predicting class without image content")
S["shortcut_top_features"] = imp.head(6).round(4).reset_index().to_dict("records")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
d = imp.head(12).iloc[::-1]
ax.barh(d.index, d["importance"], xerr=d["sd"], color="#C44E52", alpha=0.85)
ax.set_xlabel("drop in macro F1 when shuffled"); ax.set_title("(a) Which capture variables leak class?")

ax = axes[1]
labels = list(FEATURE_SETS)
vals4 = [r["macro_f1"] for r in rows4]; vals2 = [r["macro_f1"] for r in rows2]
x = np.arange(len(labels))
ax.bar(x - 0.19, vals4, 0.36, label="4-class morphology", color="#C44E52")
ax.bar(x + 0.19, vals2, 0.36, label="binary detection", color="#4C72B0")
ax.axhline(rows4[0]["baseline_macro_f1"], ls="--", lw=1, color="#888888")
ax.text(-0.4, rows4[0]["baseline_macro_f1"] + 0.01, "majority-class baseline", fontsize=8, color="#666")
for xi, v in zip(x - 0.19, vals4): ax.text(xi, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
for xi, v in zip(x + 0.19, vals2): ax.text(xi, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([l.replace(" ", "\n") for l in labels], fontsize=8)
ax.set_ylabel("macro F1"); ax.set_title("(b) Shortcut size under each normalisation policy")
ax.legend(fontsize=8); ax.set_ylim(0, 1)
fig.suptitle("Test A — how much class information is in capture conditions alone?", y=1.03, fontsize=12)
fig.tight_layout()
savefig(fig, "metadata_shortcut", "Classification from capture metadata alone, by feature set")
plt.show()

print(classification_report(y4, rows4[0]["_pred"], digits=3))

---
## Test B — Is the focus difference an artefact, or is it the crack?

`lap_var_egg` had the largest effect size of any capture variable. But Laplacian variance measures
high-frequency content, and **a crack is high-frequency content**. Measuring it over the whole egg,
cracks included, partly measures the signal rather than the acquisition.

The clean experiment: recompute focus and luminance over the egg region with the annotated crack
polygons masked out and dilated. If the class difference collapses, `lap_var_egg` was mostly signal
and is not a confound. If it survives, the eggs really were photographed differently and it is.

In [ ]:
MASK_SIDE = 512

def parse_polys(cell):
    if not isinstance(cell, str) or not cell.strip():
        return []
    return [np.asarray(o["points"], float) for o in json.loads(cell) if len(o.get("points", [])) >= 3]

def crack_free_stats(path, crack_region):
    """Focus and luminance over the egg, with the annotated crack region excluded."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return {"cf_ok": False}
    g = cv2.resize(img, (MASK_SIDE, MASK_SIDE), interpolation=cv2.INTER_AREA)

    # egg mask (same procedure as notebook 01)
    blur = cv2.GaussianBlur(g, (5, 5), 0)
    _, m = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, 8)
    egg = ((lab == 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))) if n > 1
           else np.ones_like(g, bool)).astype(bool)

    # crack mask from the polygons, dilated so the crack's halo is also removed
    crack = np.zeros((MASK_SIDE, MASK_SIDE), np.uint8)
    for p in parse_polys(crack_region):
        pts = np.round(np.column_stack([p[:, 0], p[:, 1]]) / 100.0 * MASK_SIDE).astype(np.int32)
        cv2.fillPoly(crack, [pts], 1)
    if crack.any():
        k = max(3, int(POLY_DILATE_FRAC * MASK_SIDE) | 1)
        crack = cv2.dilate(crack, np.ones((k, k), np.uint8))
    keep = egg & (crack == 0)
    if keep.sum() < 500:                       # crack covers almost the whole egg
        return {"cf_ok": False}

    lap = cv2.Laplacian(g, cv2.CV_64F)
    return {"cf_ok": True,
            "cf_lap_var": float(lap[keep].var()),
            "cf_mean_lum": float(g[keep].mean()),
            "cf_std_lum": float(g[keep].std()),
            "cf_keep_frac": float(keep.sum() / max(egg.sum(), 1))}

print(f"recomputing crack-free statistics for {len(master)} eggs ...")
cf = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(crack_free_stats)(r.path, r.crack_region) for r in master.itertuples())
cf = pd.DataFrame(cf)
aud = pd.concat([master.reset_index(drop=True), cf], axis=1)
print(f"usable: {int(aud['cf_ok'].sum())}/{len(aud)}")

In [ ]:
def kw(frame, var, by="label4"):
    gs = [g[var].dropna().values for _, g in frame.groupby(by, observed=True)]
    gs = [g for g in gs if len(g)]
    if len(gs) < 2 or np.ptp(np.concatenate(gs)) == 0:
        return 0.0, 1.0, 0.0
    H, p = stats.kruskal(*gs)
    n, k = sum(len(g) for g in gs), len(gs)
    return float(H), float(p), float((H - k + 1) / (n - k))

ok = aud[aud["cf_ok"] == True]
PAIRS = [("lap_var_egg", "cf_lap_var", "focus (Laplacian variance)"),
         ("egg_mean_lum", "cf_mean_lum", "mean luminance"),
         ("egg_std_lum", "cf_std_lum", "luminance SD")]
rows = []
for whole, free, label in PAIRS:
    H1, p1, e1 = kw(ok, whole); H2, p2, e2 = kw(ok, free)
    # a negligible or negative epsilon-squared means there was no effect to explain,
    # so the ratio is undefined rather than zero
    pct = (round(float(np.clip(100 * e2 / e1, 0, 200)), 1) if e1 > 0.01 else np.nan)
    rows.append({"measure": label,
                 "eps2_whole_egg": round(e1, 4), "p_whole_egg": p1,
                 "eps2_crack_excluded": round(e2, 4), "p_crack_excluded": p2,
                 "pct_of_effect_remaining": pct})
cmp_tbl = pd.DataFrame(rows).set_index("measure")
savetable(cmp_tbl, "t03_signal_vs_artefact",
          "Class differences in capture variables before and after excluding the annotated crack region")
S["signal_vs_artefact"] = cmp_tbl.reset_index().round(4).to_dict("records")
print(cmp_tbl.to_string())
print("\npct_of_effect_remaining near 0  -> the difference WAS the crack: genuine signal, not a confound")
print("pct_of_effect_remaining near 100 -> the eggs really were photographed differently: a CONFOUND")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.9))
for ax, (whole, free, label) in zip(axes, PAIRS):
    w = 0.36; x = np.arange(len(CFG.LABEL4))
    mw = [ok.loc[ok["label4"].eq(c), whole].median() for c in CFG.LABEL4]
    mf = [ok.loc[ok["label4"].eq(c), free].median() for c in CFG.LABEL4]
    ax.bar(x - w/2, mw, w, label="whole egg", color="#C44E52", alpha=0.9)
    ax.bar(x + w/2, mf, w, label="crack region excluded", color="#4C72B0", alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels(CFG.LABEL4, rotation=20, fontsize=8)
    r = cmp_tbl.loc[label]
    pct = r["pct_of_effect_remaining"]
    tail = "n/a" if pd.isna(pct) else f"{pct:.0f}% remains"
    ax.set_title(f"{label}\n$\\epsilon^2$ {r['eps2_whole_egg']:.3f} -> {r['eps2_crack_excluded']:.3f}"
                 f"  ({tail})", fontsize=9)
    ax.legend(fontsize=8)
fig.suptitle("Test B — separating genuine crack signal from acquisition artefact", y=1.04, fontsize=12)
fig.tight_layout()
savefig(fig, "signal_vs_artefact", "Capture-variable differences before and after excluding the annotated crack region")
plt.show()

---
## Test C — Capture session structure

Eggs were renamed by capture order, so the numeric part of `E###` is the order in which they were
photographed. If capture conditions drift smoothly with that order, there were distinct sessions;
if class also correlates with order, class and session are entangled.

This decides a concrete question for notebook 02: should the train/test split be **stratified
random** (fine if there is no session structure) or should it additionally **balance capture
blocks**, so that the test set is not dominated by one session?

In [ ]:
CAPTURE_VARS = ["min_side", "egg_mean_lum", "egg_std_lum", "lap_var_egg", "rb_ratio"]
rows = []
for v in CAPTURE_VARS:
    rho, p = stats.spearmanr(aud["capture_order"], aud[v], nan_policy="omit")
    rows.append({"variable": v, "spearman_rho": round(float(rho), 3), "p_value": float(p)})
drift = pd.DataFrame(rows).set_index("variable")
savetable(drift, "t04_capture_drift",
          "Spearman correlation between capture order and capture conditions")
S["capture_drift"] = drift.reset_index().to_dict("records")
print("drift of capture conditions over capture order:")
print(drift.to_string())

dec = pd.qcut(aud["capture_order"], 10, labels=[f"D{i}" for i in range(1, 11)])
ct = pd.crosstab(dec, aud["label4"])
chi2_o, p_o, dof_o, _ = stats.chi2_contingency(ct)
cram = math.sqrt(chi2_o / (len(aud) * (min(ct.shape) - 1)))
savetable(ct, "t05_class_by_capture_decile", "Class composition across deciles of capture order")
S["capture_order_class"] = {"chi2": round(float(chi2_o), 1), "p": float(p_o),
                            "cramers_v": round(float(cram), 3)}
print(f"\nclass x capture-order decile: chi2({dof_o}) = {chi2_o:.1f}, p = {p_o:.3g}, V = {cram:.3f}")
print("V > 0.20 means capture order and class are materially entangled.")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, v in zip(axes.ravel(), CAPTURE_VARS):
    for c in CFG.LABEL4:
        s = aud[aud["label4"].eq(c)]
        ax.scatter(s["capture_order"], s[v], s=7, alpha=0.55, color=CFG.PALETTE[c], label=c)
    roll = aud.sort_values("capture_order")[v].rolling(40, center=True, min_periods=10).median()
    ax.plot(aud.sort_values("capture_order")["capture_order"], roll, color="black", lw=1.4)
    ax.set_xlabel("capture order (egg number)"); ax.set_title(
        f"{v}   $\\rho$ = {drift.loc[v, 'spearman_rho']:+.2f}", fontsize=10)
axes.ravel()[0].legend(fontsize=7, ncol=2)

ax = axes.ravel()[5]
prop = (ct.div(ct.sum(1), axis=0) * 100)
bottom = np.zeros(len(prop))
for c in CFG.LABEL4:
    ax.bar(prop.index.astype(str), prop[c], bottom=bottom, color=CFG.PALETTE[c], label=c, width=0.8)
    bottom += prop[c].values
ax.set_ylabel("% of eggs"); ax.set_xlabel("decile of capture order")
ax.set_title(f"Class mix by capture order\nV = {cram:.3f}, p = {p_o:.1e}", fontsize=10)
ax.legend(fontsize=7, ncol=2)
fig.suptitle("Test C — did capture conditions and class drift together over the session?",
             y=1.02, fontsize=12)
fig.tight_layout()
savefig(fig, "capture_session_structure",
        "Drift of capture conditions over capture order, with class composition by decile")
plt.show()

---
## Test D — Aspect ratio and letterbox padding

Notebook 01 reported that not every image is square, and several sample images show black bars.
`crop_eggs.py` padded rather than cropped when the egg sat near an edge. Padding is a visible,
learnable artefact: if one class is padded more than another, the network can read the bars.

In [ ]:
def padding_stats(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return {"pad_ok": False}
    h, w = img.shape
    g = cv2.resize(img, (256, 256), interpolation=cv2.INTER_AREA)
    dark = g <= 8
    # a padded row/column is one that is almost entirely near-black
    row_pad = int((dark.mean(axis=1) > 0.98).sum())
    col_pad = int((dark.mean(axis=0) > 0.98).sum())
    return {"pad_ok": True, "aspect": float(w / h), "exact_square": bool(w == h),
            "pad_rows_frac": row_pad / 256, "pad_cols_frac": col_pad / 256,
            "pad_any": bool(row_pad + col_pad > 4), "dark_frac": float(dark.mean())}

pads = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(padding_stats)(p) for p in aud["path"]))
aud = pd.concat([aud, pads], axis=1)

ar = aud.groupby("label4", observed=True).agg(
    n=("aspect", "size"), exact_square=("exact_square", "sum"),
    median_aspect=("aspect", "median"), min_aspect=("aspect", "min"), max_aspect=("aspect", "max"),
    pct_padded=("pad_any", lambda s: round(100 * s.mean(), 1)),
    median_dark_frac=("dark_frac", "median")).round(3)
savetable(ar, "t06_aspect_and_padding", "Aspect ratio and letterbox padding by class")
S["aspect_padding"] = ar.reset_index().astype(str).to_dict("records")
print(ar.to_string())

H, p, e = kw(aud, "dark_frac")
print(f"\ndark (padding) fraction differs by class: H = {H:.1f}, p = {p:.3g}, eps2 = {e:.4f}")
S["padding_eps2"] = round(e, 4)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9))
ax = axes[0]
for c in CFG.LABEL4:
    ax.hist(aud.loc[aud["label4"].eq(c), "aspect"], bins=40, alpha=0.55,
            color=CFG.PALETTE[c], label=c)
ax.axvline(1.0, ls="--", color="#888888", lw=1)
ax.set_xlabel("aspect ratio (width / height)"); ax.set_ylabel("eggs")
ax.set_title("(a) Aspect ratio"); ax.legend(fontsize=8)

ax = axes[1]
data = [aud.loc[aud["label4"].eq(c), "dark_frac"].values for c in CFG.LABEL4]
bp = ax.boxplot(data, patch_artist=True, showfliers=False, widths=0.55)
try: ax.set_xticklabels(CFG.LABEL4)
except Exception: pass
for patch, c in zip(bp["boxes"], CFG.LABEL4):
    patch.set_facecolor(CFG.PALETTE[c]); patch.set_alpha(0.8)
ax.set_ylabel("fraction of near-black pixels")
ax.set_title(f"(b) Letterbox padding   $\\epsilon^2$ = {e:.3f}")
ax.tick_params(axis="x", labelrotation=15)
fig.suptitle("Test D — framing artefacts that a network could read", y=1.03, fontsize=12)
fig.tight_layout()
savefig(fig, "aspect_and_padding", "Aspect ratio and letterbox padding by class")
plt.show()

---
## Test E — Which normalisation removes the shortcut?

Test A showed how much class information capture conditions carry. This test applies candidate
pre-processing policies to the *statistics* and re-runs the probe, so the pre-processing decision in
notebook 02 is made on evidence.

Policies simulated:

1. **resize only** — absolute size information destroyed (every model does this anyway)
2. **resize + per-image intensity standardisation** — subtract the egg-region mean, divide by the
   egg-region SD; removes global brightness and contrast offsets
3. **resize + standardisation + grayscale** — additionally removes colour-temperature differences
4. **all of the above + padding removed** — simulates tight-cropping to the egg bounding box

Each policy is simulated by removing the features it would destroy. This is an upper bound on the
residual shortcut, not an exact simulation, and the notebook says so.

In [ ]:
POLICIES = {
    "none (raw)": FEATURE_SETS["all capture features"],
    "resize": [f for f in FEATURE_SETS["all capture features"] if f not in SIZE_FEATS],
    "resize + per-image standardisation":
        [f for f in FEATURE_SETS["all capture features"]
         if f not in SIZE_FEATS + ["mean_lum", "std_lum", "egg_mean_lum", "egg_std_lum",
                                   "p01_lum", "p99_lum", "lap_var_global", "lap_var_egg"]],
    "resize + standardisation + grayscale":
        [f for f in FEATURE_SETS["all capture features"]
         if f not in SIZE_FEATS + ["mean_lum", "std_lum", "egg_mean_lum", "egg_std_lum",
                                   "p01_lum", "p99_lum", "lap_var_global", "lap_var_egg",
                                   "mean_B", "mean_G", "mean_R", "rb_ratio"]],
    "all + tight crop (no padding)":
        [f for f in ["frac_clipped_hi", "frac_clipped_lo", "egg_frac_clipped_hi"]
         if f in master.columns],
}
POLICIES = {k: v for k, v in POLICIES.items() if len(v) >= 1}

pol_rows = []
for name, feats in POLICIES.items():
    r = shortcut_probe(feats, y4, name)
    pol_rows.append({k: v for k, v in r.items() if not k.startswith("_")})
pol = pd.DataFrame(pol_rows).rename(columns={"feature_set": "policy"}).set_index("policy")
pol["residual_vs_raw"] = (pol["macro_f1"] / pol.loc["none (raw)", "macro_f1"]).round(2)
savetable(pol, "t07_normalisation_policies",
          "Residual metadata shortcut under each candidate pre-processing policy")
S["normalisation_policies"] = pol.reset_index().to_dict("records")
print(pol.to_string())

fig, ax = plt.subplots(figsize=(8.5, 4.0))
d = pol.reset_index()
ax.barh(d["policy"], d["macro_f1"], color="#937860", alpha=0.9)
ax.axvline(pol["baseline_macro_f1"].iloc[0], ls="--", color="#888888", lw=1)
ax.text(pol["baseline_macro_f1"].iloc[0] + 0.005, -0.4, "majority baseline", fontsize=8, color="#666")
for i, v in enumerate(d["macro_f1"]):
    ax.text(v + 0.006, i, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlabel("macro F1 obtainable from capture conditions alone")
ax.set_title("Test E — residual shortcut after each pre-processing policy")
ax.invert_yaxis(); ax.set_xlim(0, max(0.8, d["macro_f1"].max() * 1.2))
fig.tight_layout()
savefig(fig, "normalisation_policies", "Residual metadata shortcut under candidate pre-processing policies")
plt.show()

---
## Decision and export

In [ ]:
# --- turn the evidence into the decisions notebook 02 will implement -----
dec = []
h = S["shortcut_headline_macro_f1"]
dec.append(f"Metadata-only macro F1 is {h} ({S['shortcut_band']}). "
           f"Majority baseline is {rows4[0]['baseline_macro_f1']}.")

best_pol = pol["macro_f1"].idxmin()
raw = pol.loc["none (raw)", "macro_f1"]
if pol.loc[best_pol, "macro_f1"] >= raw - 0.01:
    dec.append("No pre-processing policy materially reduces the metadata shortcut, which means the "
               "leak is not carried by brightness, colour or absolute size. Treat the capture-order "
               "structure (Test C) as the mechanism instead.")
else:
    dec.append(f"Lowest residual shortcut comes from policy: '{best_pol}' "
               f"(macro F1 {pol.loc[best_pol, 'macro_f1']}, "
               f"{pol.loc[best_pol, 'residual_vs_raw']:.0%} of raw).")

for r in S["signal_vs_artefact"]:
    pct = r["pct_of_effect_remaining"]
    if pct is None or (isinstance(pct, float) and math.isnan(pct)):
        dec.append(f"{r['measure']}: no material class effect to decompose "
                   f"(whole-egg eps2 = {r['eps2_whole_egg']}).")
        continue
    verdict = ("mostly genuine crack signal" if pct < 35
               else "mostly acquisition artefact" if pct > 65 else "mixed")
    dec.append(f"{r['measure']}: {pct:.0f}% of the class effect survives "
               f"crack exclusion -> {verdict}.")

v = S["capture_order_class"]["cramers_v"]
dec.append(f"Class vs capture-order decile V = {v}. "
           + ("Session structure is material; notebook 02 will balance capture blocks across the "
              "split and report it." if v >= 0.20 else
              "Session structure is weak; stratified random splitting is sufficient."))

dec.append(f"Padding differs by class with eps2 = {S['padding_eps2']}. "
           + ("Tight-cropping to the egg bounding box is required."
              if S["padding_eps2"] > 0.06 else
              "Padding is not a material shortcut; letterbox can be retained."))

S["decisions"] = dec
print("\n".join(f"- {d}" for d in dec))

In [ ]:
# --- persist the audited table and the digest ----------------------------
KEEP = ["egg_id", "label4", "age_class", "capture_order", "cf_ok", "cf_lap_var", "cf_mean_lum",
        "cf_std_lum", "cf_keep_frac", "aspect", "exact_square", "pad_rows_frac", "pad_cols_frac",
        "pad_any", "dark_frac"]
aud[[c for c in KEEP if c in aud.columns]].to_csv(PROC / "confound_audit.csv", index=False)
print(f"wrote {PROC / 'confound_audit.csv'}")

S["figures"], S["tables"] = FIGS, TABS
def _safe(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
(OUT / "confound_summary.json").write_text(json.dumps(S, indent=2, default=_safe))

L = ["=" * 70, "NOTEBOOK 01b DIGEST -- CONFOUND AND LEAKAGE AUDIT", "=" * 70,
     f"generated: {S['generated_at']}", ""]
L.append(f"TEST A  metadata-only macro F1 = {h}  [{S['shortcut_band'].upper()}]")
for r in S["shortcut_probe"]:
    L.append(f"   {r['task']:<20s} {r['feature_set']:<32s} macroF1={r['macro_f1']:<6} "
             f"bal_acc={r['balanced_acc']}")
L.append("   top leaking features:")
for r in S["shortcut_top_features"]:
    L.append(f"      {r['feature']:<22s} {r['importance']:+.4f}")
L.append("")
L.append("TEST B  signal vs artefact (% of class effect surviving crack exclusion)")
for r in S["signal_vs_artefact"]:
    _p = r["pct_of_effect_remaining"]
    _t = "n/a" if (_p is None or (isinstance(_p, float) and math.isnan(_p))) else f"{_p:.0f}% remains"
    L.append(f"   {r['measure']:<28s} eps2 {r['eps2_whole_egg']:.3f} -> "
             f"{r['eps2_crack_excluded']:.3f}   ({_t})")
L.append("")
L.append("TEST C  capture session structure")
for r in S["capture_drift"]:
    L.append(f"   drift {r['variable']:<16s} rho={r['spearman_rho']:+.3f}  p={r['p_value']:.2e}")
L.append(f"   class x capture-order decile: V = {S['capture_order_class']['cramers_v']}")
L.append("")
L.append(f"TEST D  padding eps2 = {S['padding_eps2']}")
L.append("")
L.append("TEST E  residual shortcut by policy")
for r in S["normalisation_policies"]:
    L.append(f"   {r['policy']:<40s} macroF1={r['macro_f1']}")
L.append("")
L.append("DECISIONS")
for d in S["decisions"]:
    L.append(f"   - {d}")
L.append("=" * 70)
digest = "\n".join(L)
(OUT / "digest.txt").write_text(digest)
print(digest)